# Update a person

`PATCH /admin/people/{id}` changes team, role, and `manager_id`. Unlike a
policy body edit, a person write always bumps the version. The snapshot keeps
`manager_id`. A "who reports to whom" answer is read from that field, in code,
not from prose.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [ ]:
PERSON_ID = "emp-900002"
KEY = f"raw/hr/person/{PERSON_ID}.json"
PAYLOAD = {
    "person_id": PERSON_ID,
    "email": "emp-900002@example.com",
    "full_name": "Demo Person",
    "team": "people",
    "role": "People Partner",
    "region": "DE",
    "manager_id": None,
    "native_acl": ["hr_employee"],
}
hr_demo.delete_if_present("person", PERSON_ID)
created = hr_demo.admin("POST", "/admin/people", PAYLOAD)
print(created["person_id"], "version", created["version"], created["native_acl"])


In [ ]:
import json

def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    if not all(meta.get(name) for name in hr_demo.META_KEYS):
        return False
    if meta.get(hr_demo.META_VERSION) != str(created["version"]):
        return False
    raw = hr_demo.object_body(KEY)
    record = json.loads(raw)
    leaked = set(record) & hr_demo.FORBIDDEN
    return raw == hr_demo.canonical(record) and not leaked

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "person snapshot")
record = json.loads(hr_demo.object_body(KEY))
assert record == hr_demo.admin("GET", f"/people/{PERSON_ID}")
print(sorted(record))
print(hr_demo.snapshot(KEY)["Metadata"])


## Edit team, role, and manager


In [ ]:
import json

others = [
    item["person_id"]
    for item in hr_demo.admin("GET", "/people")["items"]
    if item["person_id"] != PERSON_ID
]
if not others:
    raise SystemExit("the directory has no other person to report to")
manager_id = others[0]
before_version = hr_demo.snapshot(KEY)["Metadata"][hr_demo.META_VERSION]
updated = hr_demo.admin(
    "PATCH",
    f"/admin/people/{PERSON_ID}",
    {"team": "platform", "role": "Software Engineer", "manager_id": manager_id},
)
assert updated["manager_id"] == manager_id
assert updated["team"] == "platform"
assert updated["version"] == int(before_version) + 1

def moved() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    if head["Metadata"].get(hr_demo.META_VERSION) != str(updated["version"]):
        return False
    record = json.loads(hr_demo.object_body(KEY))
    return record["manager_id"] == manager_id and record["team"] == "platform"

hr_demo.wait_until(moved, hr_demo.WRITE_WAIT, "person update")
record = json.loads(hr_demo.object_body(KEY))
print(record["full_name"], "reports to", record["manager_id"], "on", record["team"])
print(hr_demo.snapshot(KEY)["Metadata"])


## Negative case


In [ ]:
missing = hr_demo.expect_status(
    "PATCH",
    "/admin/people/emp-000000",
    404,
    {"team": "platform"},
)
print("missing", missing.status_code)


## Cleanup


In [ ]:
hr_demo.delete_if_present("person", PERSON_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)
